# Builds on Signal Construction and produces long/short/do-nothing signal for past 24 months

##### Step 1: Set datetimes for each month for last 2 years
##### Step 2: Run signal construction code for each month
##### Step 3: Make signal matrix with monthly buy and sell signals

# 1. Get array of Datetime objects for the last 2 years on a monthly frequency

In [1]:
# Our usual imports 
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import datetime
import pandas as pd

# Define the universe of stocks

univ = ['DUOL', 'PEP', 'KO', 'JNJ', 'IBM', 'MCD', 'WMT', 'PG', 'DIS', 'VZ', 'TMUS', 'COST', 'NKE', 'UPS', 'AVGO', 'TXN', 
        'BAC', 'CAT', 'ORCL', 'HOOD', 'COIN', 'TER', 'PYPL', 'ADBE', 'CRM', 'INTU', 'NFLX', 'AMZN', 'GOOGL', 'META', 'AAPL', 'INTC']

In [2]:
# Make datetime arrays consisting of monthly dates for the last 2 years
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=365*2)
dates = pd.date_range(start=start_date, end=end_date, freq='ME')
dates

DatetimeIndex(['2024-03-31 12:54:28.925329', '2024-04-30 12:54:28.925329',
               '2024-05-31 12:54:28.925329', '2024-06-30 12:54:28.925329',
               '2024-07-31 12:54:28.925329', '2024-08-31 12:54:28.925329',
               '2024-09-30 12:54:28.925329', '2024-10-31 12:54:28.925329',
               '2024-11-30 12:54:28.925329', '2024-12-31 12:54:28.925329',
               '2025-01-31 12:54:28.925329', '2025-02-28 12:54:28.925329',
               '2025-03-31 12:54:28.925329', '2025-04-30 12:54:28.925329',
               '2025-05-31 12:54:28.925329', '2025-06-30 12:54:28.925329',
               '2025-07-31 12:54:28.925329', '2025-08-31 12:54:28.925329',
               '2025-09-30 12:54:28.925329', '2025-10-31 12:54:28.925329',
               '2025-11-30 12:54:28.925329', '2025-12-31 12:54:28.925329',
               '2026-01-31 12:54:28.925329', '2026-02-28 12:54:28.925329'],
              dtype='datetime64[ns]', freq='ME')

# 2. Importing signal construction code with few tweaks to produce signals for each month for last 2 years

In [3]:
# defining our functions that we are going to use in our loop 

def get_momentum_scores(prices, lookback=252, skip=21):
    """12-month momentum excluding the most recent month."""

    if len(prices) < lookback + skip:
        return {}

    momentum_window = prices.iloc[-(lookback + skip):-skip]

    first = momentum_window.iloc[0]
    last = momentum_window.iloc[-1]

    momentum_scores = (last - first) / first # for now, this is how I calculate momentum score, I will change when I learn more

    return momentum_scores.dropna().to_dict()


def get_reversal_scores(prices, window):
    """Mean-reversion (reverse momentum) over a given window (in trading days)."""

    if len(prices) < window:
        return {}

    reversal_window = prices.iloc[-window:]

    first = reversal_window.iloc[0]
    last = reversal_window.iloc[-1]

    reversal_scores = -((last - first) / first)

    return reversal_scores.dropna().to_dict()


# 
# def get_earnings_scores(tickers):
#    earnings_scores = {}
#    
#    for ticker in tickers:
#        stock = yf.Ticker(ticker)
#        earnings = stock.get_earnings_dates(limit=40)  # Get the past 40 quarters of earnings data
#        if not earnings.empty and len(earnings) >= 8:
#            # Calculate earnings score as the average percentage change in earnings over the past 4 quarters
#            earnings["surprise"] = (
#            (earnings["Reported EPS"] - earnings["EPS Estimate"]) /
#                earnings["EPS Estimate"].abs()
#            )
#            average_surprise = earnings['surprise'].mean()
#            earnings_scores[ticker] = average_surprise
#            
#    return earnings_scores

### Earnings Function get revamped to avoid look-ahead bias

In [5]:

def get_earnings_history(tickers):

    earnings_data = {}

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        earnings = stock.get_earnings_dates(limit=40)

        if not earnings.empty:
            earnings["surprise"] = (
                (earnings["Reported EPS"] - earnings["EPS Estimate"]) /
                earnings["EPS Estimate"].abs()
            )

            earnings_data[ticker] = earnings["surprise"]

    return earnings_data

def compute_earnings_score(earnings_series, current_date, lookback=4):

    past = earnings_series[earnings_series.index <= current_date]

    if len(past) < lookback:
        return None

    return past.tail(lookback).mean()

def build_earnings_factor(tickers, dates):

    earnings_data = get_earnings_history(tickers)

    factor = pd.DataFrame(index=dates, columns=tickers)

    for date in dates:
        for ticker in tickers:

            if ticker in earnings_data:
                score = compute_earnings_score(
                    earnings_data[ticker],
                    date
                )

                factor.loc[date, ticker] = score

    return factor

earnings_scores = build_earnings_factor(univ, dates)


TypeError: Invalid comparison between dtype=datetime64[ns, America/New_York] and Timestamp

In [19]:
# empty dateframe with dates as index and columns as tickers, to store signals for each date and ticker
signals_df = pd.DataFrame(index=dates, columns=univ)

prices = yf.download(univ, start="2022-01-01", end="2026-02-28", auto_adjust=False)['Adj Close']
# Earnings changes only quarterly so compute once and reuse inside the loop

earnings_scores = build_earnings_factor(univ, dates)

sorted_pscores = {}
for date in dates:
    prices_until_date = prices.loc[:date]

    # Need enough history for the momentum window (12m) plus the skip month
    if len(prices_until_date) < 252 + 21:
        continue

    # Momentum Stuff
    momentum_scores = get_momentum_scores(prices_until_date)
    momentum_s = pd.Series(momentum_scores)
    momentum_z = (momentum_s - momentum_s.mean()) / momentum_s.std()
    
    # Reveral Stuff
    reversal_scores_1m = get_reversal_scores(prices_until_date, 21)
    reversal_scores_2m = get_reversal_scores(prices_until_date, 42)
    reversal_scores_3m = get_reversal_scores(prices_until_date, 63)
    # Combine the reversal scores into a single score by averaging the 3, 2, and 1 month scores
    combined_reversal = (
        pd.Series(reversal_scores_1m) +
        pd.Series(reversal_scores_2m) +
        pd.Series(reversal_scores_3m)
    ) / 3
    reversal_z = (combined_reversal - combined_reversal.mean()) / combined_reversal.std()
    
    # Earnings Stuff
    earnings_z = (pd.Series(earnings_scores) - pd.Series(earnings_scores).mean()) / pd.Series(earnings_scores).std()
        
    # Compute pscore
    pscores = {}
    for ticker in univ:
        m_score = 35 * momentum_z.get(ticker, 0)
        r_score = 50 * reversal_z.get(ticker, 0)
        e_score = 15 * earnings_z.get(ticker, 0)
        pscores[ticker] = m_score + r_score + e_score

    sorted_by_score = dict(sorted(pscores.items(), key=lambda item: item[1], reverse=True))
    sorted_pscores[date] = sorted_by_score

    # Generate long/short/neutral signals (top/bottom 20%)
    n = len(sorted_by_score)
    n_rank = max(1, int(np.ceil(n * 0.2)))
    long_tickers = list(sorted_by_score.keys())[:n_rank]
    short_tickers = list(sorted_by_score.keys())[-n_rank:]  

    for ticker in univ:
        if ticker in long_tickers:
            signals_df.loc[date, ticker] = 1
        elif ticker in short_tickers:
            signals_df.loc[date, ticker] = -1
        else:
            signals_df.loc[date, ticker] = 0



[*********************100%***********************]  32 of 32 completed


TypeError: Invalid comparison between dtype=datetime64[ns, America/New_York] and Timestamp

In [6]:
signals_df

,DUOL,PEP,KO,JNJ,IBM,MCD,WMT,PG,DIS,VZ,...,PYPL,ADBE,CRM,INTU,NFLX,AMZN,GOOGL,META,AAPL,INTC
2024-03-31 12:15:24.249111,1,0,0,0,0,0,0,0,-1,0,...,-1,1,0,1,0,0,0,1,1,1
2024-04-30 12:15:24.249111,0,-1,-1,0,1,0,0,0,0,0,...,-1,1,1,0,1,0,-1,1,0,1
2024-05-31 12:15:24.249111,1,0,0,0,0,0,-1,0,0,0,...,0,1,1,1,0,0,-1,1,-1,1
2024-06-30 12:15:24.249111,1,0,0,0,0,0,-1,0,1,0,...,1,-1,0,0,0,0,-1,0,-1,1
2024-07-31 12:15:24.249111,1,0,0,-1,-1,0,0,0,1,0,...,-1,-1,0,0,0,0,1,1,-1,0
2024-08-31 12:15:24.249111,-1,0,-1,-1,0,-1,0,0,1,0,...,-1,-1,0,0,0,1,1,1,0,1
2024-09-30 12:15:24.249111,-1,0,0,0,0,-1,0,0,0,0,...,-1,1,0,0,0,0,1,0,0,0
2024-10-31 12:15:24.249111,-1,0,1,0,1,0,0,0,-1,0,...,-1,1,-1,0,0,0,0,0,0,-1
2024-11-30 12:15:24.249111,-1,0,1,0,0,0,0,0,-1,0,...,0,0,-1,0,0,0,1,1,0,-1
2024-12-31 12:15:24.249111,0,0,0,0,0,0,0,0,-1,1,...,0,0,-1,0,0,-1,-1,0,-1,1


##### Some debugging to see how my signals change over time and if my factors are actually playing a part or not

In [11]:
signals_df.nunique()

DUOL     3
PEP      3
KO       3
JNJ      2
IBM      3
MCD      3
WMT      3
PG       2
DIS      3
VZ       3
TMUS     3
COST     3
NKE      3
UPS      3
AVGO     3
TXN      3
BAC      3
CAT      3
ORCL     3
HOOD     3
COIN     3
TER      3
PYPL     3
ADBE     3
CRM      3
INTU     3
NFLX     3
AMZN     3
GOOGL    3
META     3
AAPL     3
INTC     3
dtype: int64

In [8]:
signal_changes = signals_df.diff().abs().sum(axis=1)
print(signal_changes)

2024-03-31 12:15:24.249111     0
2024-04-30 12:15:24.249111    18
2024-05-31 12:15:24.249111    12
2024-06-30 12:15:24.249111    14
2024-07-31 12:15:24.249111    14
2024-08-31 12:15:24.249111    12
2024-09-30 12:15:24.249111    14
2024-10-31 12:15:24.249111    12
2024-11-30 12:15:24.249111    12
2024-12-31 12:15:24.249111    20
2025-01-31 12:15:24.249111    14
2025-02-28 12:15:24.249111    22
2025-03-31 12:15:24.249111    10
2025-04-30 12:15:24.249111    12
2025-05-31 12:15:24.249111    20
2025-06-30 12:15:24.249111    20
2025-07-31 12:15:24.249111    14
2025-08-31 12:15:24.249111    14
2025-09-30 12:15:24.249111     8
2025-10-31 12:15:24.249111    10
2025-11-30 12:15:24.249111     6
2025-12-31 12:15:24.249111    14
2026-01-31 12:15:24.249111    12
2026-02-28 12:15:24.249111    10
Freq: ME, dtype: object


In [10]:
signal_changes = (signals_df != signals_df.shift()).sum(axis=1)
signal_changes

2024-03-31 12:15:24.249111    32
2024-04-30 12:15:24.249111    18
2024-05-31 12:15:24.249111    12
2024-06-30 12:15:24.249111    13
2024-07-31 12:15:24.249111    12
2024-08-31 12:15:24.249111    10
2024-09-30 12:15:24.249111    13
2024-10-31 12:15:24.249111    12
2024-11-30 12:15:24.249111    12
2024-12-31 12:15:24.249111    16
2025-01-31 12:15:24.249111    13
2025-02-28 12:15:24.249111    18
2025-03-31 12:15:24.249111    10
2025-04-30 12:15:24.249111    12
2025-05-31 12:15:24.249111    19
2025-06-30 12:15:24.249111    19
2025-07-31 12:15:24.249111    14
2025-08-31 12:15:24.249111    14
2025-09-30 12:15:24.249111     8
2025-10-31 12:15:24.249111    10
2025-11-30 12:15:24.249111     6
2025-12-31 12:15:24.249111    12
2026-01-31 12:15:24.249111    12
2026-02-28 12:15:24.249111    10
Freq: ME, dtype: int64